In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [2]:
import pandas as pd
import numpy as np
import csv
import pickle
import pickle as pkl
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import openai
import time
import sys
import pickle as pkl
import os


!pip install ipython-autotime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.6 MB/s eta 0:00:00


In [1]:


!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "4_Bundle Generation"



Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 36 (delta 1), reused 15 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 23.98 KiB | 3.00 MiB/s, done.
Resolving deltas: 100% (1/1), done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 67 bytes | 67.00 KiB/s, done.
remote: Enumerating objects: 106, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 106 (delta 16), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (106/106), 1.45 MiB | 5.83 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Updating files: 100% (110/110), done.
Successfully created: /content/drive/MyDrive/EGPO/filter_results/


In [ ]:

import os

path = "/content/drive/MyDrive/baselines/"

try:
    os.makedirs(path, exist_ok=True)
    print(f"Successfully created: {path}")
except Exception as e:
    print(f"An error occurred: {e}")



In [ ]:
from google.colab import userdata
my_secret_key = userdata.get('API_KEY')

if my_secret_key:
  print("Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")


open_secret_key = userdata.get('open_router')

if open_secret_key:
  print("OpenRouter Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")


from openai import OpenAI

# client = OpenAI(
#     # This is the default and can be omitted
#     api_key = my_secret_key,
# )


import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=open_secret_key,
)

async_client = AsyncOpenAI(api_key = my_secret_key,
)  # make sure this is your actual key


In [ ]:
def get_zero_shot_prompts(data_info):

    split_test = data_info.split('|split|')
    empty = ""
    for i in range(len(split_test)):
        empty += "product" + str(i + 1) + ". " + split_test[i] + "\n"

    test_prompts = """A bundle can be a set of alternative or complementary products that are purchased with a certain intent.\nPlease detect bundles from a sequence of products. Each bundle must contain multiple products.\nDetect bundles for the below product sequence:\n\n""" + empty + "\n"





    output_format = """Try to infer any intents that may exist in the list of products if purchased together, each intent should help form a bundle with associated products.
Each bundle should have more than a single product. Use product numbers rather than item names. There could be a single or multiple bundles present.

**## OUTPUT FORMAT:**
After your intent analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that contains your bundles.

**JSON Schema:**
```json
{{
  "bundle1": ["product number", "product number", "product number", "..."],
  "bundle2": ["product number", "product number"],
  "bundleN": ["product number", "product number", "..."]
}}
````
Think step-by-step."
"""
    test_prompts += output_format
    return test_prompts

def get_few_shot_prompts(data_info, example_info, example_session_bundles, item_titles):
    start_char_code = ord('a')
    alphabet = ["(" + chr(start_char_code + i) + ") " for i in range(26)]

    ground_bundles = example_session_bundles

    amount_of_bundles = len(ground_bundles)
    empty_ground = ""
    for i in range(len(ground_bundles)):
        ground_items = ""
        ground_keys = ground_bundles[i][1].split(",")
        for j in range(len(ground_keys)):
            ground_items += "- " +item_titles[ground_keys[j]] + "\n"

        ground_str = f"""Example Bundle {i+1}:
Intent: {ground_bundles[i][0]}
Items:
{ground_items}
"""
        empty_ground += ground_str



    example_from_training = example_info.split('|split|')
    number_of_example_items = len(example_from_training)
    example_empty = ""
    for i in range(len(example_from_training)):
        example_empty += alphabet[i] + example_from_training[i] + "\n"

    split_test = data_info.split('|split|')
    empty = ""
    for i in range(len(split_test)):
        empty += "product" + str(i + 1) + ". " + split_test[i] + "\n"

    test_prompts = f"""A bundle can be a set of alternative or complementary products that are purchased with a certain intent.
Please detect bundles from a sequence of products. Each bundle must contain multiple products.
Carefully observe the example where on a list of {number_of_example_items} items has been grouped into {amount_of_bundles} bundles:


Example List:
{example_empty}
{empty_ground}

Your task is to detect bundles for the below product sequence:

{empty}

"""





    output_format = """Try to infer any intents that may exist in the list of products if purchased together, each intent should help form a bundle with associated products.
Each bundle should have more than a single product. Use product numbers rather than item names. There could be a single or multiple bundles present.

**## OUTPUT FORMAT:**
After your intent analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that contains your bundles.

**JSON Schema:**
```json
{{
  "bundle1": ["product number", "product number", "product number", "..."],
  "bundle2": ["product number", "product number"],
  "bundleN": ["product number", "product number", "..."]
}}
````
Think step-by-step."
"""
    test_prompts += output_format
    return test_prompts

def few_shot_prompt_from_id(key_id, test_set, training_set, session_bundles_deduplication, item_titles, top_K):
    nearest_neighbour_id = top_K[key_id][0]

    test_example = test_set[key_id]
    example_info = training_set[nearest_neighbour_id]
    session_bundle_example = session_bundles_deduplication[nearest_neighbour_id]

    return get_few_shot_prompts(test_example, example_info, session_bundle_example, item_titles)


In [8]:
async def run_zero_and_few_shot_tests(path, model):

    dataset = path.split("/")[5]

    domain = path.split("/")[6]

    item_titles_path = "item_titles.npy"
    session_bundles_deduplication_path = "session_bundles_deduplication.npy"
    session_items = "session_items.npy"
    topK_related_sessions_path = "TopK_related_sessions.npy"
    training_set_path = "training_set.npy"
    test_set_path = "test_set.npy"

    item_titles = np.load(path + item_titles_path, allow_pickle=True).tolist()
    session_bundles_deduplication = np.load(path + session_bundles_deduplication_path, allow_pickle=True).tolist()
    session_items = np.load(path + session_items, allow_pickle=True).tolist()
    topK_related_sessions = np.load(path + topK_related_sessions_path, allow_pickle=True).tolist()
    training_set = np.load(path + training_set_path, allow_pickle=True).tolist()
    test_set = np.load(path + test_set_path, allow_pickle=True).tolist()

    test_keys = list(test_set.keys())

    zero_shot_prompt_list = [{"prompts": get_zero_shot_prompts(test_set[i])} for i in test_keys]

    few_shot_prompt_list = [{"prompts": few_shot_prompt_from_id(i, test_set, training_set, session_bundles_deduplication, item_titles, topK_related_sessions)} for i in test_keys]

    system_message = "You are an **Expert E-commerce Analyst and Product Bundler**. Your task is to receive a sequence of products and accurately organise them into logical, desirable bundles."

    zero_shot_responses = await openai_request(zero_shot_prompt_list, model=model, system=system_message)
    few_shot_responses = await openai_request(few_shot_prompt_list, model=model, system=system_message)

    with open(f"/content/drive/MyDrive/baselines/{dataset}/{domain}/" + f'{model}_zero_shot_responses.pkl', 'wb') as f:
        pkl.dump(zero_shot_responses, f)

    with open(f"/content/drive/MyDrive/baselines/{dataset}/{domain}/" + f'{model}_few_shot_responses.pkl', 'wb') as f:
        pkl.dump(few_shot_responses, f)

    return zero_shot_responses, few_shot_responses

async def openrouter_run_zero_and_few_shot_tests(path):

    dataset = path.split("/")[5]

    domain = path.split("/")[6]

    item_titles_path = "item_titles.npy"
    session_bundles_deduplication_path = "session_bundles_deduplication.npy"
    session_items = "session_items.npy"
    topK_related_sessions_path = "TopK_related_sessions.npy"
    training_set_path = "training_set.npy"
    test_set_path = "test_set.npy"

    item_titles = np.load(path + item_titles_path, allow_pickle=True).tolist()
    session_bundles_deduplication = np.load(path + session_bundles_deduplication_path, allow_pickle=True).tolist()
    session_items = np.load(path + session_items, allow_pickle=True).tolist()
    topK_related_sessions = np.load(path + topK_related_sessions_path, allow_pickle=True).tolist()
    training_set = np.load(path + training_set_path, allow_pickle=True).tolist()
    test_set = np.load(path + test_set_path, allow_pickle=True).tolist()

    test_keys = list(test_set.keys())

    zero_shot_prompt_list = [{"prompts": get_zero_shot_prompts(test_set[i])} for i in test_keys]

    few_shot_prompt_list = [{"prompts": few_shot_prompt_from_id(i, test_set, training_set, session_bundles_deduplication, item_titles, topK_related_sessions)} for i in test_keys]

    system_message = "You are an **Expert E-commerce Analyst and Product Bundler**. Your task is to receive a sequence of products and accurately organise them into logical, desirable bundles."

    baseline_models = [
        "google/gemini-2.0-flash-001",
        "anthropic/claude-3-5-haiku",
        "meta-llama/llama-3.3-70b-instruct",
        "mistralai/mistral-small-24b-instruct-2501"
    ]
    names = ["gemini", "claude", "llama", "mistral"]

    batch_sizes = [40, 40, 10, 10]

    for i in range(2,3):
        zero_shot_responses = await run_experiment(zero_shot_prompt_list, model_id=baseline_models[i], system=system_message, batch_size=batch_sizes[i])
        few_shot_responses = await run_experiment(few_shot_prompt_list, model_id=baseline_models[i], system=system_message, batch_size=batch_sizes[i])

        with open(f"/content/drive/MyDrive/baselines/{dataset}/{domain}/" + f'{names[i]}_zero_shot_responses.pkl', 'wb') as f:
            pkl.dump(zero_shot_responses, f)

        with open(f"/content/drive/MyDrive/baselines/{dataset}/{domain}/" + f'{names[i]}_few_shot_responses.pkl', 'wb') as f:
            pkl.dump(few_shot_responses, f)

    # return zero_shot_responses, few_shot_responses

SyntaxError: unterminated f-string literal (detected at line 80) (ipython-input-3019519626.py, line 80)

In [ ]:
from tqdm.asyncio import tqdm_asyncio

async def single_request(user, model, system=None, seed_value=None):

    if system:
        message = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    else:
        message = [{"role": "user", "content": user}]

    # Reimplemented the robust retry loop with exponential backoff
    for delay_secs in (2**x for x in range(0, 3)):
        try:
            response = await async_client.chat.completions.create(
                model=model #"gpt-4.1-mini",
                messages=message,
                temperature=0,
                max_tokens=2000,
                seed=seed_value
            )
            return response.choices[0].message.content.strip()
        except openai.OpenAIError as e:
            randomness_collision_avoidance = random.randint(0, 1000) / 1000.0
            sleep_dur = delay_secs + randomness_collision_avoidance
            print(f"Error: {e}. Retrying in {round(sleep_dur, 2)} seconds.")
            await asyncio.sleep(sleep_dur)

    # Return None if all retries fail
    return None


async def openai_request(prompts, model, system=None, batch_size=128, delay=0):

    results = []

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        tasks = [
            single_request(d["prompts"], model, system=system, seed_value=42)
            for j, d in enumerate(batch)
        ]

        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)
        print(f"✅ Sending batch {i // batch_size + 1} — sleeping for {delay}s...\n")
        await asyncio.sleep(delay)

    return results
from tqdm.asyncio import tqdm_asyncio

async def openrouter_request(user, model_id, system=None):
    if system:
        message = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    else:
        message = [{"role": "user", "content": user}]

    # Robust retry loop with exponential backoff
    for delay_secs in (2**x for x in range(0, 3)):
        try:
            response = await async_client.chat.completions.create(
                model=model_id, # Now dynamic!
                messages=message,
                temperature=0,
                max_tokens=2000, # Adjust based on bundle length
                # Optional: extra_body is where OpenRouter specific features go
                extra_body={
                    "provider": {"require_parameters": True}
                }
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            randomness_collision_avoidance = random.randint(0, 1000) / 1000.0
            sleep_dur = delay_secs + randomness_collision_avoidance
            print(f"Error with {model_id}: {e}. Retrying in {round(sleep_dur, 2)}s.")
            await asyncio.sleep(sleep_dur)

    return None


async def run_experiment(prompts, model_id, system=None, batch_size=20):
    """
    Unified experiment runner with a real-time progress bar.
    It still respects the Semaphore 'bouncer' you set up globally.
    """
    results = []
    print(f"🚀 Initializing experiment for: {model_id}")

    # 1. Create all tasks immediately.
    # Your 'async with sem:' inside single_request will handle the
    # actual throttling so you don't hit rate limits.
    tasks = [
        openrouter_request(d["prompts"], model_id, system=system)
        for d in prompts
    ]

    # 2. Use tqdm_asyncio.gather to run all tasks with a live progress bar.
    # This replaces the manual 'for' loop and 'asyncio.gather' batches.
    results = await tqdm_asyncio.gather(
        *tasks,
        desc=f"📊 {model_id.split('/')[-1]}", # Shows model name (e.g., 'llama-3.3-70b')
        total=len(tasks)
    )

    print(f"✅ {model_id} — All {len(results)} requests completed.\n")
    return results

In [ ]:
def load_pkl(file_path):
    """Safely loads a pickle file."""
    try:
        with open(file_path, 'rb') as f:
            return pkl.load(f)
    except FileNotFoundError:
        print(f"❌ ERROR: File not found at {file_path}")
        return None
    except Exception as e:
        print(f"❌ ERROR loading pickle file: {e}")
        return None



def extract_json_simple_replace(response_text):
    if response_text is None:
        return None

    try:
        # 1. Use a case-insensitive split or check for the separator
        if "===JSON_START===" not in response_text:
            # Fallback: Try to find the first '{' anyway
            json_part = response_text
        else:
            json_part = response_text.split("===JSON_START===")[1]

        # 2. Find the boundaries
        first_brace = json_part.find('{')
        last_brace = json_part.rfind('}')

        if first_brace == -1 or last_brace == -1:
            return None

        # 3. Extract and clean
        json_string = json_part[first_brace : last_brace + 1].strip()

        # 4. Parse
        return json.loads(json_string)

    except Exception as e:
        print(f"Extraction error: {e}")
        return None

In [5]:
electronic_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/electronic/'
clothing_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/clothing/'
food_bundlerec_path = '/content/LLM4BEAR/4_Bundle Generation/data/bundlerec/food/'

electronic_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/electronic/'
clothing_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/clothing/'
food_llm4bear_path = '/content/LLM4BEAR/4_Bundle Generation/data/llm4bear/food/'


In [ ]:

# ==========================================
# EXECUTE ORDER 66
# ==========================================

print("🚀 STARTING FULL BATCH EXECUTION (6 DATASETS)...\n")

# --- 1. ELECTRONIC (BundleRec) ---

# print("--------------------------------------------------\n")
# print("💻 [1/6] Running ELECTRONIC BundleRec...\n")
# await openrouter_run_zero_and_few_shot_tests(electronic_bundlerec_path)
# print("✅ Electronic BundleRec Finished.")




# # --- 2. CLOTHING (BundleRec) ---

# print("\n--------------------------------------------------\n")
# print("👕 [2/6] Running CLOTHING BundleRec...\n")
# await openrouter_run_zero_and_few_shot_tests(clothing_bundlerec_path)
# print("✅ Clothing BundleRec Finished.")


# # # --- 3. FOOD (BundleRec) ---

# print("\n--------------------------------------------------\n")
# print("🍔 [3/6] Running FOOD BundleRec...\n")
# await openrouter_run_zero_and_few_shot_tests(food_bundlerec_path)
# print("✅ Food BundleRec Finished.")


# # --- 4. ELECTRONIC (LLM4BEAR) ---

# print("\n--------------------------------------------------\n")
# print("💻 [4/6] Running ELECTRONIC LLM4BEAR...\n")
# await openrouter_run_zero_and_few_shot_tests(electronic_llm4bear_path)
# print("✅ Electronic LLM4BEAR Finished.")


# # --- 5. CLOTHING (LLM4BEAR) ---

# print("\n--------------------------------------------------\n")
# print("👕 [5/6] Running CLOTHING LLM4BEAR...\n")
# await openrouter_run_zero_and_few_shot_tests(clothing_llm4bear_path)
# print("✅ Clothing LLM4BEAR Finished.")

# # --- 6. FOOD (LLM4BEAR) ---

# print("\n--------------------------------------------------\n")
# print("🍔 [6/6] Running FOOD LLM4BEAR...\n")
# await openrouter_run_zero_and_few_shot_tests(food_llm4bear_path)
# print("✅ Food LLM4BEAR Finished.")


print("\n🎉🎉🎉 ALL 6 EXPERIMENTS COMPLETED 🎉🎉🎉")

In [ ]:
models = ["gpt-4o-mini", "gpt-4.1-mini"]

# ==========================================
# EXECUTE ORDER 66
# ==========================================

# for model in models:

#     print("🚀 STARTING FULL BATCH EXECUTION (6 DATASETS)...\n")

#     # # --- 1. ELECTRONIC (BundleRec) ---

#     print("--------------------------------------------------\n")
#     print("💻 [1/6] Running ELECTRONIC BundleRec...\n")
#     brec_elec_zero, brec_elec_few = await run_zero_and_few_shot_tests(electronic_bundlerec_path, model=model)
#     print("✅ Electronic BundleRec Finished.")




#     # --- 2. CLOTHING (BundleRec) ---

#     print("\n--------------------------------------------------\n")
#     print("👕 [2/6] Running CLOTHING BundleRec...\n")
#     brec_clot_zero, brec_clot_few = await run_zero_and_few_shot_tests(clothing_bundlerec_path, model=model)
#     print("✅ Clothing BundleRec Finished.")


#     # # --- 3. FOOD (BundleRec) ---

#     print("\n--------------------------------------------------\n")
#     print("🍔 [3/6] Running FOOD BundleRec...\n")
#     brec_food_zero, brec_food_few = await run_zero_and_few_shot_tests(food_bundlerec_path, model=model)
#     print("✅ Food BundleRec Finished.")


#     # # --- 4. ELECTRONIC (LLM4BEAR) ---

#     print("\n--------------------------------------------------\n")
#     print("💻 [4/6] Running ELECTRONIC LLM4BEAR...\n")
#     bear_elec_zero, bear_elec_few = await run_zero_and_few_shot_tests(electronic_llm4bear_path, model=model)
#     print("✅ Electronic LLM4BEAR Finished.")


#     # --- 5. CLOTHING (LLM4BEAR) ---

#     print("\n--------------------------------------------------\n")
#     print("👕 [5/6] Running CLOTHING LLM4BEAR...\n")
#     bear_clot_zero, bear_clot_few = await run_zero_and_few_shot_tests(clothing_llm4bear_path, model=model)
#     print("✅ Clothing LLM4BEAR Finished.")

#     # --- 6. FOOD (LLM4BEAR) ---

#     print("\n--------------------------------------------------\n")
#     print("🍔 [6/6] Running FOOD LLM4BEAR...\n")
#     bear_food_zero, bear_food_few = await run_zero_and_few_shot_tests(food_llm4bear_path, model=model)
#     print("✅ Food LLM4BEAR Finished.")


#     print("\n🎉🎉🎉 ALL 6 EXPERIMENTS COMPLETED 🎉🎉🎉")
#     print()
